# Bioinformatics pipeline testing

This notebook implements DADA2 and VSEARCH _de novo_ pipelines for four different sets of amplicon data (three real and one simulated), covering all steps through to taxonomy assignment. It also loads in denoised data from the MetaBEAT pipeline to test alternative taxonomic assignments. The DADA2 workflow is adapted from the [NEOF version](https://github.com/khmaher/HPC_dada2/tree/main?tab=readme-ov-file), and the VSEARCH workflow is based on a published [GitHub pipeline](https://github.com/torognes/vsearch) and [VSEARCH documentation](https://torognes.github.io/vsearch/), both modified for this project.

The pipeline includes:
- Removal of Ns and primers (cutadapt)
- Read quality control, filtering, and trimming (filterAndTrim)
- Error modelling, dereplication, denoising, read merging and chimera removal (DADA2 & VSEARCH)

Taxonomic assignment includes:
- RDP, BLAST, and MapSeq using a custom reference database combining MetaFishLib (fish) and MIDORI2 (non-fish).
- We also have MetaBEAT taxonomic assignment, performed externally using an older reference database.
- taxonomizr is used to condense BLAST results

**NOTE:** There are numerous places where the test dataset name must be changed to either Marchamley, Ringtrial_Sean, Windermere_2017 or Simulated. Please search and replace.

## 00. Set-up

Install packages, load required functions and get data.

In [0]:
source("Scripts/00_setup.R") #packages and functions

In [0]:
%sh

apt-get -q update

# cutadapt
apt-get -q install -y cutadapt
#pip install -q cutadapt
#cutadapt --version

# vsearch install
apt-get -q install -y vsearch

# BLAST+ installed
apt-get -q install -y ncbi-blast+

# MAPseq
cd /dbfs/tmp # temp directory
wget https://github.com/jfmrod/MAPseq/releases/download/v2.1.1/mapseq-2.1.1-linux.tar.gz # download
tar -xvzf mapseq-2.1.1-linux.tar.gz # extract

## check install
cd /dbfs/tmp/mapseq-2.1.1-linux
./mapseq

In [0]:
test_data_name <- "Windermere_2017" # change here to update test data (same name as folder)
test_data_loc <- file.path("Data", "Raw", test_data_name)
results_loc = paste("Results/", test_data_name)
print(head(list.files(test_data_loc)))

In [0]:
manifest <- make_manifest(test_data_loc)

dir.create(
  file.path("Data", "Temp", test_data_name),
  recursive = TRUE,
  showWarnings = FALSE
)

write.csv(
  manifest,
  file.path("Data", "Temp", test_data_name, "manifest.csv"),
  row.names = FALSE
)

head(manifest)

## 01. Removing Ns

In [0]:
source("Scripts/01_remove_Ns.R")

## 02. Identify and remove primers

In [0]:
# Riaz 2011

FWD <- "ACTGGGATTAGATACCCC"
REV <- "TAGAACAGGCTCCTCTAG"

In [0]:
# these would be options in actual pipeline

minimum <- 60 #Minimum read length cutoff. half the length of the amplicon
copies <- 2 #Number of copies of a primer to be removed as sometimes dulication can occur. Recommended minimum is 2

In [0]:
source("Scripts/02_primer_removal.R")

## 03. Generate quality plots

In [0]:
source("Scripts/03_raw_quality_plots.R")

In the plots below:
- Grey-scale heatmap shows the frequency of each quality score along the read lengths - looking for over 30 ish. 
- Green line is median quality score. 
- Orange line are quartiles. 
- Red line at the bottom represent the proportion of reads of that particular length. 

Our reads on average are approx 100bp long after primers removed in the first two samples, which is about right for these primers.

In [0]:
print(plotQualityProfile(fnFs.cut[1:2]))

In [0]:
print(plotQualityProfile(fnRs.cut[1:2]))

## 04. Cleaning the data (filterAndTrim)

Parameters for filterAndTrim include:
- **maxN** - after truncation, sequences with more then X Ns will be disgarded
- **truncQ** - Truncate reads at the first instance of a quality score less then or equal to X
- **rm.phix** - Discard reads that match against the phiX genome
- **maxEE** - After truncation, reads with higher than X expected errors will be discarded
- **minLen** - Remove reads with length less than 60 (note these should gave already been removed by cutadapt)
- **multithread** - input files are filtered in parallel (logical)
- **truncLen** - controls where each read is cut (truncated) based on its length

In [0]:
truncLen=c(80,95) # where the quality dropped in previous plots
maxEE <-  c(2,2) #maxEE value 
truncQ <- 2 #truncQ value
minLen <- 50 #Impose a minimum length cutoff

# other options in NEOF pipeline are subset (only run a portion of the data) and marker (for better labelling when running multiple markers)

In [0]:
source("Scripts/04_filterAndTrim.R")

In [0]:
plotQualityProfile(fnFs.filtN[5:6])

In [0]:
plotQualityProfile(fnRs.filtN[1:2])

## 05. Generate error model

In [0]:
source("Scripts/05_generate_error_model.R")

In the plots below:
- Error rates for each possible transition (e.g. A->C, A->G) are shown
- Red line = expected based on quality score
- Black line = estimate 
- Black dots = observed 

We want black lines and black dots to match. We also want to see a rough negative correlation. 

If it looks weird, can increase the number of bases the function is using (default is 100 million).

In [0]:
plotErrors(errF, nominalQ = TRUE)


In [0]:
plotErrors(errR, nominalQ = TRUE)

Common to see 'hook' between 30 - 40 with NovaSeq and MiniSeq data when visualising in DADA2. Couple of forums have discussion on this topic. Not to be of too much concern unless seeing any weird results.

## 06. Deplication, merging and chimera removal

### DADA2

In [0]:
source("Scripts/06a_derep_DADA2.R")

### VSEARCH de novo

In [0]:
%sh
bash Scripts/06b_derep_VSEARCH.sh Windermere_2017 #change to test data

## 07. Sequence tracking & formatting

In [0]:
source("Scripts/07a_sequence_tracking.R")

Input and filtered columns will be the same for both pipelines, as they were done in the filter and trim step. They are not repeated below.

Let's look at just the VSEARCH steps post filterAndTrim. They are in a different order the DADA2 (merging happens first in VSEARCH) so please take note of the column names.

In [0]:
%sh
bash Scripts/07b_sequence_tracking.sh Windermere_2017 #change to test data

In [0]:
source("Scripts/07c_format_VSEARCH.R")

The number of sequences should look roughly similar between the two methods.

We now need to get the MetaBEAT data in a similar format to the DADA2 outputs.

In [0]:
%sh
python -u Scripts/07d_format_MetaBEAT.py # parameters to change at top of script

Check out data outputs.

In [0]:
# load dataset
ASV_tab_DADA2 <- read.table(
  file = file.path("Data", "Processed", test_data_name, "06_ASV_counts_DADA2.tsv"),
  header = TRUE,
  sep = "\t",
  row.names = 1,
  check.names = FALSE
)

In [0]:
ASV_tab_VSEARCH <- read.table(
  file = file.path("Data", "Processed", test_data_name, "06b_ASV_counts_VSEARCH.tsv"),
  header = TRUE,
  sep = "\t",
  row.names = 1,
  check.names = FALSE
)

In [0]:
ASV_seq_DADA2 <- readDNAStringSet(file.path("Data", "Processed", test_data_name, "06_ASV_seqs_DADA2.fasta"))
ASV_seq_VSEARCH <- readDNAStringSet(file.path("Data", "Processed", test_data_name, "06b_ASV_seqs_VSEARCH.fasta"))

## 08. Assign taxonomy

### RDP

In [0]:
source("Scripts/08a_assign_taxonomy_RDP_DADA2.R") # included MeatBEAT file processing in here too

In [0]:
source("Scripts/08c_assign_taxonomy_RDP_VSEARCH.R")

In [0]:
source("Scripts/08e_assign_taxonomy_RDP_MetaBEAT.R") # no data for simulated data

### BLAST

Tidy MetaFishLib for BLAST.

In [0]:
%python
import pandas as pd
# Load your database
mfl_db = pd.read_csv("/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/EA_custom/EA_fish_database.riaz.csv")

# Clean sequences + names
mfl_db = mfl_db.dropna(subset=["gbAccession", "nucleotides"])
mfl_db["seq"] = mfl_db["nucleotides"].str.upper().str.replace(r"[^ACGT]", "", regex=True)
mfl_db["species"] = mfl_db["sciNameValid"].str.replace(r"[^A-Za-z ]", "", regex=True)

# Write FASTA
with open("Data/Databases/EA_custom/EA_fish_riaz.fasta", "w") as f:
    for _, row in mfl_db.iterrows():
        header = f">{row['gbAccession']} {row['species']}"
        f.write(header + "\n")
        f.write(row["seq"] + "\n")

In [0]:
%sh
# to make BLAST formatted database from curated Meta Fish Lb
makeblastdb -in /Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/EA_custom/EA_fish_riaz.fasta -dbtype nucl -out Data/Databases/EA_custom/EA_fish_riaz_db/EA_fish_riaz_db

Now run blast.

In [0]:
%sh
bash Scripts/08b_assign_taxonomy_BLAST_DADA2.sh Windermere_2017 # change to test data

In [0]:
%sh
bash Scripts/08d_assign_taxonomy_BLAST_VSEARCH.sh Windermere_2017 # change to test data

In [0]:
%sh
bash Scripts/08e_assign_taxonomy_BLAST_MetaBEAT.sh Windermere_2017 # change to test data, no data for simulated

#### Condense BLAST taxonomy

Condense taxonomy (lowest common ancestor) for BLAST outputs. 

In [0]:
min_pident <- 95
min_length <- 88 # 20% within 110 amplicon length
top_frac <- 0.95

In [0]:
source("Scripts/09_condensing_taxonomy_DADA2.R")

In [0]:
source("Scripts/09b_condensing_taxonomy_VSEARCH.R")

In [0]:
source("Scripts/09c_condensing_taxonomy_MetaBEAT.R") # no data for simulated

In [0]:
%sh
python -u Scripts/09d_tidy_MetaBEAT_blast.py # parameters to change at top of script, no data for simulated

### MAPSeq

Let's format our reference database to the correct mapseq format.

In [0]:
%python
import csv
from collections import defaultdict

# ---- file paths ----
input_csv = "/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/EA_custom/EA_fish_database.riaz.csv"

base_dir = "/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/EA_custom/"

fasta_out = base_dir + "EA_riaz_mapseq.fasta"
tax_out = base_dir + "EA_riaz_mapseq.tax"
cluster_out = base_dir + "EA_riaz_mapseq.fasta.mscluster"

# ---- storage ----
records = []
species_to_ids = defaultdict(list)

def clean(x):
    if x is None:
        return "unknown"
    x = x.strip()
    return x.replace(" ", "_") if x and x != "NA" else "unknown"

# ---- load csv ----
with open(input_csv, newline='', encoding="utf-8-sig") as csvfile:
    reader = csv.DictReader(csvfile)

    for row in reader:
        seq_id = row["gbAccession"].strip()
        sequence = row["nucleotides"].strip().upper()

        kingdom = clean(row.get("kingdom"))
        phylum = clean(row.get("phylum"))
        class_ = clean(row.get("class"))
        order = clean(row.get("order"))
        family = clean(row.get("family"))
        genus = clean(row.get("genus"))
        species = clean(row.get("sciNameValid"))

        # ensure species format
        if "_" not in species:
            species = f"{genus}_sp"

        records.append((seq_id, sequence, kingdom, phylum, class_, order, family, genus, species))
        species_to_ids[species].append(seq_id)

print(f"Loaded {len(records)} records")

# ---- write fasta ----
with open(fasta_out, "w") as f:
    for r in records:
        seq_id, sequence = r[0], r[1]
        f.write(f">{seq_id}\n{sequence}\n")

print("FASTA complete")

# ---- write taxa ----
with open(tax_out, "w") as f:

    # header          Kingdom   Phylum    Class     Order     Family    Genus     Species
    f.write("#cutoff: 0.00:0.08 0.70:0.35 0.70:0.35 0.70:0.35 0.80:0.25 0.92:0.08 0.95:0.05\n") # took from github values
    f.write("#name: MetaFish\n")
    f.write("#levels: Kingdom Phylum Class Order Family Genus Species\n")

    for r in records:
        seq_id, _, kingdom, phylum, class_, order, family, genus, species = r

        taxonomy = f"{kingdom};{phylum};{class_};{order};{family};{genus};{species}"

        f.write(f"{seq_id}\t{taxonomy}\n")

print("TAX file complete")

In [0]:

%sh
# dataset name 
test_data_name="Windermere_2017" # remember to change

# input ASVs
input_fasta_dada2="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/06_ASV_seqs_DADA2.fasta"
input_fasta_vsearch="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/06b_ASV_seqs_VSEARCH.fasta"
input_fasta_MetaBEAT="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/06_ASV_seqs_MetaBEAT.fasta"

# MAPseq database 
db_dir="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/EA_custom"

db_fasta="${db_dir}/EA_riaz_mapseq.fasta"
db_tax="${db_dir}/EA_riaz_mapseq.tax"

# output 
out_file_dada2="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/08_mapseq_output_dada2_EA_riaz.mseq"
out_file_vsearch="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/08_mapseq_output_vsearch_EA_riaz.mseq"
out_file_MetaBEAT="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/08_mapseq_output_MetaBEAT_EA_riaz.mseq"

# dada2
/dbfs/tmp/mapseq-2.1.1-linux/mapseq \
    -nthreads 16 \
    -tophits 80 \
    -topotus 40 \
    "$input_fasta_dada2" \
    "$db_fasta" "$db_tax" \
    > "$out_file_dada2"

# vsearch
/dbfs/tmp/mapseq-2.1.1-linux/mapseq \
    -nthreads 16 \
    -tophits 80 \
    -topotus 40 \
    "$input_fasta_vsearch" \
    "$db_fasta" "$db_tax" \
    > "$out_file_vsearch"

# MetaBEAT
/dbfs/tmp/mapseq-2.1.1-linux/mapseq \
    -nthreads 16 \
    -tophits 80 \
    -topotus 40 \
    "$input_fasta_MetaBEAT" \
    "$db_fasta" "$db_tax" \
    > "$out_file_MetaBEAT"

In [0]:
%sh
python -u Scripts/09e_tidy_mapseq.py # parameters to change at top of script

### VSEARCH

In [0]:
%sh

# used dada2 fasta for this
fasta_file="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/EA_custom/EA_fish_database_VSEARCH.fasta"
vsearch_db_output="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/EA_custom/vsearch_indexed_reference"

vsearch --makeudb_usearch "$fasta_file" --output "$vsearch_db_output" --threads 12

In [0]:
%sh
# dataset name 
test_data_name="Windermere_2017" # remember to change

# input ASVs
input_fasta_dada2="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/06_ASV_seqs_DADA2.fasta"
input_fasta_vsearch="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/06b_ASV_seqs_VSEARCH.fasta"
input_fasta_MetaBEAT="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/06_ASV_seqs_MetaBEAT.fasta"

# output files
out_file_dada2="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/08_vsearch_output_dada2_EA_riaz.tsv"
out_file_vsearch="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/08_vsearch_output_vsearch_EA_riaz.tsv"
out_file_MetaBEAT="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Processed/${test_data_name}/08_vsearch_output_MetaBEAT_EA_riaz.tsv"

# vsearch database 
db_dir="/Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/EA_custom/vsearch_indexed_reference"

#dada2
vsearch --usearch_global "$input_fasta_dada2" --db "$db_dir" --threads 4 --userout "$out_file_dada2" --strand both --id 0.60 --maxaccepts 5 --userfields query+target+id+qcov --top_hits_only

#vsearch
vsearch --usearch_global "$input_fasta_vsearch" --db "$db_dir" --threads 4 --userout "$out_file_vsearch" --strand both --id 0.60 --maxaccepts 5 --userfields query+target+id+qcov --top_hits_only

#metabeat
vsearch --usearch_global "$input_fasta_MetaBEAT" --db "$db_dir" --threads 4 --userout "$out_file_MetaBEAT" --strand both --id 0.60 --maxaccepts 5 --userfields query+target+id+qcov --top_hits_only

In [0]:
%sh

for dataset in Marchamley RingTrial_Sean Windermere_2017 Simulated
do
    for file in Data/Processed/"$dataset"/08_vsearch_output*.tsv
    do
        [[ "$file" == *_lca.tsv ]] && continue

        outfile="${file%.tsv}_lca.tsv"

        python Scripts/09e_lca_pid_qcov_filtering.py "$file" > "$outfile"
    done
done

End of pipeline. Please continue to the next notebook to explore comparisons.